# Cloud Native User API — Integration Tests

This notebook tests all endpoints of the FastAPI User API.

**Prerequisites:** The API must be running and accessible at `BASE_URL`.
- Local: `docker compose up --build`
- AWS: Set `BASE_URL` to the ALB DNS name after `helm install`

In [ ]:
import requests
import time
import json

# Change this to your ALB DNS when testing against AWS
BASE_URL = "http://localhost:8000"

def pp(resp):
    """Pretty-print a response."""
    print(f"{resp.request.method} {resp.url}")
    print(f"Status: {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text[:500])
    print()

print(f"Target: {BASE_URL}")

## 1. Health & Readiness

In [ ]:
# Root endpoint — should return company name and status
resp = requests.get(f"{BASE_URL}/")
pp(resp)
assert resp.status_code == 200
data = resp.json()
assert "company" in data
assert data["status"] == "running"
print(f"Company: {data['company']}")

# Liveness probe
resp = requests.get(f"{BASE_URL}/healthz")
pp(resp)
assert resp.status_code == 200
assert resp.json()["status"] == "ok"

# Readiness probe (checks DB + Redis)
resp = requests.get(f"{BASE_URL}/readyz")
pp(resp)
assert resp.status_code == 200
assert resp.json()["status"] == "ready"

print("All health checks passed.")

## 2. Create Users (POST /users)

In [ ]:
# Create first user
resp = requests.post(f"{BASE_URL}/users", json={"name": "Alice", "email": "alice@example.com"})
pp(resp)
assert resp.status_code == 200
user1 = resp.json()
assert user1["name"] == "Alice"
assert user1["email"] == "alice@example.com"
assert "id" in user1

# Create second user
resp = requests.post(f"{BASE_URL}/users", json={"name": "Bob", "email": "bob@example.com"})
pp(resp)
assert resp.status_code == 200
user2 = resp.json()
assert user2["name"] == "Bob"

print(f"Created user IDs: {user1['id']}, {user2['id']}")

## 3. Get User by ID (GET /users/{id})

In [ ]:
# Fetch user by ID
resp = requests.get(f"{BASE_URL}/users/{user1['id']}")
pp(resp)
assert resp.status_code == 200
assert resp.json()["name"] == "Alice"

# Non-existent user should return 404
resp = requests.get(f"{BASE_URL}/users/99999")
pp(resp)
assert resp.status_code == 404

print("GET /users/{id} tests passed.")

## 4. List All Users (GET /users)

In [ ]:
resp = requests.get(f"{BASE_URL}/users")
pp(resp)
assert resp.status_code == 200
users = resp.json()
assert isinstance(users, list)
assert len(users) >= 2
names = [u["name"] for u in users]
assert "Alice" in names
assert "Bob" in names

print(f"Total users: {len(users)}")

## 5. Redis Cache Behavior

The API caches responses in Redis with a 60-second TTL.
Second requests should be noticeably faster (served from cache).

In [ ]:
# First call (may or may not be cached from previous cells)
# Wait for cache to expire to get a clean test
print("Testing cache behavior for GET /users/{id}...")

# Cold call — hits DB
start = time.time()
resp1 = requests.get(f"{BASE_URL}/users/{user1['id']}")
cold_ms = (time.time() - start) * 1000

# Warm call — should hit Redis cache
start = time.time()
resp2 = requests.get(f"{BASE_URL}/users/{user1['id']}")
warm_ms = (time.time() - start) * 1000

assert resp1.json() == resp2.json()
print(f"Cold call:  {cold_ms:.1f} ms")
print(f"Warm call:  {warm_ms:.1f} ms")
print(f"Speedup:    {cold_ms / warm_ms:.1f}x" if warm_ms > 0 else "")

print()
print("Testing cache behavior for GET /users...")

start = time.time()
resp3 = requests.get(f"{BASE_URL}/users")
list_cold_ms = (time.time() - start) * 1000

start = time.time()
resp4 = requests.get(f"{BASE_URL}/users")
list_warm_ms = (time.time() - start) * 1000

assert resp3.json() == resp4.json()
print(f"Cold call:  {list_cold_ms:.1f} ms")
print(f"Warm call:  {list_warm_ms:.1f} ms")

print("\nCache tests passed.")

## 6. Cache Invalidation

Creating a new user should invalidate the `users:all` cache.

In [ ]:
# Get current user list (populates cache)
before = requests.get(f"{BASE_URL}/users").json()
print(f"Users before: {len(before)}")

# Create a new user (should invalidate users:all cache)
resp = requests.post(f"{BASE_URL}/users", json={"name": "Charlie", "email": "charlie@example.com"})
assert resp.status_code == 200
pp(resp)

# Get user list again (should include new user, not stale cache)
after = requests.get(f"{BASE_URL}/users").json()
print(f"Users after:  {len(after)}")
assert len(after) == len(before) + 1

new_names = [u["name"] for u in after]
assert "Charlie" in new_names

print("\nCache invalidation test passed.")

## 7. OpenAPI Docs

FastAPI auto-generates Swagger UI at `/docs` and OpenAPI schema at `/openapi.json`.

In [ ]:
# Swagger UI
resp = requests.get(f"{BASE_URL}/docs")
assert resp.status_code == 200
print(f"GET /docs -> {resp.status_code}")

# OpenAPI schema
resp = requests.get(f"{BASE_URL}/openapi.json")
assert resp.status_code == 200
schema = resp.json()
print(f"API title:   {schema['info']['title']}")
print(f"API version: {schema['info']['version']}")
print(f"Paths:       {list(schema['paths'].keys())}")

print("\nAll tests passed!")